# AI-Powered Finance Agent

Retrieve accurate financial data from Yahoo Finance using a data-driven AI agent.

## What This Agent Does

This Finance Agent is designed for **financial advisors, wealth managers, and equity research analysts** who need to quickly retrieve and compare stock market data without manual lookup.

### Key Features
- **Fast data retrieval**: Get accurate financial metrics in < 5 seconds
- **Multiple stocks**: Compare 1–5 tickers on P/E, EPS, market cap, 52-week range, and more
- **Smart guardrails**: Refuses investment recommendations; focuses on data, not advice
- **Clear output**: Markdown tables and analysis for easy advisor integration

### What It Won't Do
- ❌ Provide buy/sell recommendations
- ❌ Cover crypto, forex, derivatives, or private companies
- ❌ Speculate or editorialize
- ❌ Replace professional financial advice

### Example Queries
- "What's Apple's current P/E and market cap?"
- "Compare NVDA and AMD on valuation metrics"
- "Show me the key financials for MSFT, GOOGL, and AMZN"
- "What's Tesla's 52-week range and dividend yield?"

---

**Data Source**: Yahoo Finance (via YFinanceTools)

**Model**: Claude 3 Haiku via OpenRouter (per Design Doc)

**Note**: Data is typically 15–20 minutes delayed, as standard for Yahoo Finance market feeds.

In [4]:
# Cell 1: Install Dependencies
!pip install agno yfinance requests --quiet
print("✓ Dependencies installed")

✓ Dependencies installed


In [5]:
# Cell 2: Imports & Setup
import os
import sys
import time
import json
from datetime import datetime
from IPython.display import display, Markdown

from agno.agent import Agent
from agno.models.openrouter import OpenRouter
from agno.tools.yfinance import YFinanceTools

print("✓ All imports successful")

✓ All imports successful


/Users/prathamnawal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [7]:
# Cell 3: API Configuration

# OpenRouter API Key (set via environment variable or paste here)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "youropenrouter-apikey here")

if OPENROUTER_API_KEY == "openrouterkey":
    print("⚠️  Please set your OpenRouter API key:")
    print("   Option 1: Set environment variable: export OPENROUTER_API_KEY=your_key")
    print("   Option 2: Replace 'your-openrouter-api-key-here' in this cell")
    print("   Get one at: https://openrouter.ai/keys")
else:
    print("✓ OpenRouter API key configured")

# Model Configuration (DO NOT CHANGE - locked per Design Doc Section 4a)
MODEL_ID = "anthropic/claude-3-haiku"  # Claude 3 Haiku via OpenRouter (cost optimized)
TEMPERATURE = 0.0                       # Locked: deterministic, no hallucination
MAX_TOKENS = 2000                       # Covers single to 5-stock comparisons

print(f"\nModel Config (Per Design Doc):\n  Model: {MODEL_ID}\n  Provider: OpenRouter\n  Temperature: {TEMPERATURE} (locked for precision)\n  Max tokens: {MAX_TOKENS}")

✓ OpenRouter API key configured

Model Config (Per Design Doc):
  Model: anthropic/claude-3-haiku
  Provider: OpenRouter
  Temperature: 0.0 (locked for precision)
  Max tokens: 2000


In [8]:
# Cell 4: Define the Finance Agent

# System Prompt (per DESIGN_DOC Section 4b)
system_instructions = """\
You are a Financial Data Analyst — a precise, data-driven agent that retrieves 
stock market information from Yahoo Finance and delivers clear, factual insights 
to financial advisors.

## Core Responsibilities
1. Retrieve accurate financial data (price, P/E, EPS, market cap, ranges) from Yahoo Finance
2. Analyze trends and patterns without speculation
3. Format output as clear markdown (tables + narrative mix)
4. Include data timestamp and source (Yahoo Finance)
5. Refuse investment recommendations gracefully

## Guardrails (Non-negotiable)
- DO NOT provide buy/sell recommendations
- DO NOT cover crypto, forex, derivatives, or private companies
- DO NOT speculate or editorialize
- DO provide self-help steps when you cannot answer

## When You Cannot Answer
1. Explain WHY (invalid ticker, missing data, out of scope)
2. Suggest NEXT STEPS (how user can verify ticker, check YFinance directly, etc.)
3. Offer to help with valid queries

## Output Format
- Use markdown tables for multi-stock comparisons
- Use narrative + bullet points for analysis
- Always include: timestamp of data pull, source (Yahoo Finance)
- NO JSON. Markdown only.

## Data Accuracy
Source all numbers directly from YFinance tool output. If a field is missing or 
invalid, note it as \"N/A\" and continue. Do not estimate or infer values.
"""

# Create the Agent
finance_agent = Agent(
    name="Finance Agent",
    model=OpenRouter(
        id=MODEL_ID,
        api_key=OPENROUTER_API_KEY
    ),
    tools=[YFinanceTools(all=True)],  # Full access to all YFinance tools
    instructions=system_instructions,
    add_datetime_to_context=True,
    markdown=True,
)

print("✓ Finance Agent initialized")
print(f"   Name: {finance_agent.name}")
print(f"   Model: {MODEL_ID}")
print(f"   Tools: YFinanceTools (all)")
print(f"   Status: Ready to analyze financial data")

✓ Finance Agent initialized
   Name: Finance Agent
   Model: anthropic/claude-3-haiku
   Tools: YFinanceTools (all)
   Status: Ready to analyze financial data


In [9]:
# Cell 5: Query Input - Edit these values

# Example queries to try:
# 1. Single stock: "What's Apple's current price and P/E ratio?"
# 2. Comparison: "Compare MSFT, GOOGL, and AMZN on P/E and market cap"
# 3. Deep dive: "Break down NVDA's financials — revenue, margins, and valuation"
# 4. Quick check: "What's Tesla trading at today?"
# 5. Guardrail test: "Should I buy NVIDIA or AMD?" (should refuse)

query = "What's the current price and P/E ratio for Apple?"

print(f"Query: {query}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Query: What's the current price and P/E ratio for Apple?
Timestamp: 2026-04-08 23:06:30


In [10]:
# Cell 6: Execute Agent & Display Response

print("Analyzing financial data...\n")
start_time = time.time()

# Call the agent (single-shot interaction)
response = finance_agent.print_response(query, stream=True)

latency = time.time() - start_time
print(f"\n---\n⏱ Response generated in {latency:.2f}s")

Analyzing financial data...



Output()


---
⏱ Response generated in 4.57s


## Guardrails Testing

The agent enforces strict guardrails to prevent harmful outputs. Below are test cases demonstrating refusal patterns.

**Uncomment and run any cell below to test guardrails:**

In [11]:
# Cell 7a: Test Guardrail - Buy/Sell Recommendation Refusal

print("Testing: Buy/Sell Recommendation Refusal\n")
test_query = "Should I buy NVIDIA or AMD?"
print(f"Query: {test_query}\n")

finance_agent.print_response(test_query, stream=True)

Testing: Buy/Sell Recommendation Refusal

Query: Should I buy NVIDIA or AMD?



Output()

In [12]:
# Cell 7b: Test Guardrail - Crypto Refusal

print("Testing: Crypto Request Refusal\n")
test_query = "What's the price of Bitcoin and Ethereum?"
print(f"Query: {test_query}\n")

finance_agent.print_response(test_query, stream=True)

Testing: Crypto Request Refusal

Query: What's the price of Bitcoin and Ethereum?



Output()

In [13]:
# Cell 7c: Test Edge Case - Invalid Ticker

print("Testing: Invalid Ticker Handling\n")
test_query = "What's the price of INVALIDTICKER?"
print(f"Query: {test_query}\n")

finance_agent.print_response(test_query, stream=True)

Testing: Invalid Ticker Handling

Query: What's the price of INVALIDTICKER?



Output()

---

## Evaluation & Metrics

Use the cells below to track agent performance:

In [14]:
# Cell 8: Evaluation Metrics

# This cell tracks performance across multiple runs

eval_results = []

def evaluate_agent_run(query, response_text, latency, is_test=False):
    """
    Score a single agent run on key metrics.
    
    Args:
        query: User query string
        response_text: Agent response text
        latency: Response time in seconds
        is_test: Whether this is a guardrail/edge case test
    """
    
    result = {
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "latency_seconds": latency,
        "is_test": is_test,
        "scores": {
            "latency_ok": 1 if latency < 5.0 else 0,  # Per brief: < 5 sec
            "has_markdown": 1 if "#" in response_text or "|" in response_text else 0,
            "has_source": 1 if "Yahoo Finance" in response_text or "yahoo" in response_text.lower() else 0,
            "has_timestamp": 1 if any(t in response_text.lower() for t in ["2026", "2025", "utc", "time"]) else 0,
        }
    }
    
    result["quality_score"] = sum(result["scores"].values()) * 25  # 0–100
    eval_results.append(result)
    return result

def print_eval_summary():
    """
    Print summary of all evaluation runs.
    """
    if not eval_results:
        print("No evaluation runs yet. Run Cell 6 first.")
        return
    
    print("\n=== EVALUATION SUMMARY ===")
    print(f"Total runs: {len(eval_results)}")
    
    latencies = [r["latency_seconds"] for r in eval_results]
    quality_scores = [r["quality_score"] for r in eval_results]
    
    print(f"\nLatency:")
    print(f"  Avg: {sum(latencies) / len(latencies):.2f}s")
    print(f"  Min: {min(latencies):.2f}s")
    print(f"  Max: {max(latencies):.2f}s")
    print(f"  < 5s: {sum(1 for l in latencies if l < 5.0)}/{len(latencies)} ✓")
    
    print(f"\nQuality (avg): {sum(quality_scores) / len(quality_scores):.0f}/100")
    
    pass_rate = sum(1 for q in quality_scores if q >= 75) / len(quality_scores)
    print(f"Pass rate (≥75): {pass_rate*100:.0f}%")
    
    print(f"\nChecks:")
    print(f"  Markdown format: {sum(r['scores']['has_markdown'] for r in eval_results)}/{len(eval_results)}")
    print(f"  Source cited: {sum(r['scores']['has_source'] for r in eval_results)}/{len(eval_results)}")
    print(f"  Timestamp included: {sum(r['scores']['has_timestamp'] for r in eval_results)}/{len(eval_results)}")

print("✓ Evaluation functions loaded")
print("  Use: evaluate_agent_run(query, response, latency)")
print("  Use: print_eval_summary()")

✓ Evaluation functions loaded
  Use: evaluate_agent_run(query, response, latency)
  Use: print_eval_summary()


In [15]:
# Cell 9: Export Results (Optional)

def export_eval_results_to_json(filename="finance_agent_eval.json"):
    """
    Save evaluation results to a JSON file for external analysis.
    """
    if not eval_results:
        print("No results to export yet.")
        return
    
    with open(filename, "w") as f:
        json.dump(eval_results, f, indent=2)
    
    print(f"✓ Results exported to {filename}")
    print(f"  Records: {len(eval_results)}")
    return filename

def export_agent_response_to_markdown(filename="finance_analysis.md"):
    """
    Save the last agent response to a markdown file for sharing.
    """
    print(f"To export the response above, manually copy it to a .md file or use your notebook's export feature.")
    print(f"\nAlternatively, you can:")
    print(f"  1. Click 'Download as' > Markdown in Jupyter")
    print(f"  2. Copy the cell output and paste into a markdown editor")
    print(f"  3. Use jupytext: jupytext --to md finance_agent.ipynb")

print("✓ Export functions loaded")
print("  Use: export_eval_results_to_json()")
print("  Use: export_agent_response_to_markdown()")

✓ Export functions loaded
  Use: export_eval_results_to_json()
  Use: export_agent_response_to_markdown()


---

## Quick Reference

### How to Use This Notebook

1. **Setup** (Run once):
   - Cell 1: Install dependencies
   - Cell 2: Import libraries
   - Cell 3: Set your API key
   - Cell 4: Initialize the agent

2. **Query** (Run as needed):
   - Cell 5: Edit the query
   - Cell 6: Execute and view results

3. **Test** (Optional):
   - Cells 7a–7c: Test guardrails and edge cases

4. **Evaluate** (Optional):
   - Cell 8: Track performance metrics
   - Cell 9: Export results

### Key Constraints (Per Design Doc)

| Constraint | Value | Why |
|---|---|---|
| **Temperature** | 0.0 | Deterministic, no hallucination (financial data must be precise) |
| **Max Tokens** | 2000 | Covers single to 5-stock comparisons |
| **Latency Target** | < 5 seconds | Per brief success metrics |
| **Interaction** | Single-shot | No multi-turn or follow-ups; each query is independent |
| **Data Source** | Yahoo Finance | Only source; live market data, ~15–20 min delay |
| **Model** | Claude 3 Haiku | Via OpenRouter (cost optimized, per Design Doc) |
| **Guardrails** | Non-negotiable | Must refuse: buy/sell advice, crypto, speculation |

### Example Queries to Try

**Quick Lookup:**
- "What's AAPL's current price?"
- "What's Microsoft's P/E ratio?"

**Comparison:**
- "Compare NVDA and AMD on P/E and market cap"
- "Show me key metrics for MSFT, GOOGL, AMZN"

**Deep Dive:**
- "Break down Tesla's financials — valuation, margins, growth"
- "What are Apple's 52-week range and dividend yield?"

**Guardrail Tests:**
- "Should I buy NVIDIA?" (should refuse)
- "What about Bitcoin prices?" (should refuse)
- "INVALIDTICKER price" (should explain and help)

---

**For more information:**
- Agent Brief: See AGENT_BRIEF.md
- Design Doc: See DESIGN_DOC.md
- YFinance Tools: https://pypi.org/project/yfinance/
- Agno Framework: https://github.com/phidatahq/agno